# State of the Data 3: Timeliness
# Notebook 1: Preparation

## Set up the notebook environment

In [1]:
# Install dependencies

import pandas as pd
import numpy as np
import noteql
import requests
import json
import concurrent.futures
import time
from pathlib import Path
import os, sys
## import pyarrow.parquet as pa
import re, csv, math, io, shutil
from urllib.parse import quote
import urllib.request
import pyarrow as pa
import pyarrow.parquet as pq
from bs4 import BeautifulSoup
from functools import reduce
from urllib.parse import urlparse
import pickle 


# Restart postgres to make sure any existing connections get dropped
!sudo service postgresql restart
session = noteql.Session(datasette_url='https://datasette.tables.iatistandard.org/iati.json', connect_args={'connect_timeout': 1000})

In [4]:
# Ensure tqdm always shows the plain console progress bar instead of the Jupyter widget.
# Reason: the notebook widget can be buggy/slow in some environments (e.g. Deepnote),
# so we reset any previously imported tqdm modules and force the standard version.

os.environ["TQDM_NOTEBOOK"] = "0"  # never use notebook widgets

# Purge any previously loaded tqdm modules (kills tqdm.notebook if it was imported)
for m in list(sys.modules.keys()):
    if m.startswith("tqdm"):
        del sys.modules[m]

from tqdm.std import tqdm  # <-- use this tqdm everywhere below

## Downloads



#### Investigate subelements of organisation that have datetime elements i.e. budgets

As a result of this we decided not to include organisation budget data in the analysis.

In [43]:
## Total budget
response = requests.get('https://datasette.tables.iatistandard.org/iati.json?sql=select+count+%28distinct+prefix%29+from+organisation_totalbudget')
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Number of organisations with at least one total budget element:{total_count}")

## Recipient org budget
response = requests.get('https://datasette.tables.iatistandard.org/iati.json?sql=SELECT+COUNT%28DISTINCT+prefix%29+from+organisation_recipientorgbudget')
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Number of organisations with at least one recipient org budget element:{total_count}")

## Recipient region budget
response = requests.get('https://datasette.tables.iatistandard.org/iati.json?sql=SELECT+COUNT%28DISTINCT+prefix%29+from+organisation_recipientregionbudget')
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Number of organisations with at least one recipient region budget element:{total_count}")

## Recipient country budget
response = requests.get('https://datasette.tables.iatistandard.org/iati.json?sql=SELECT+COUNT%28DISTINCT+prefix%29+from+organisation_recipientcountrybudget')
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Number of organisations with at least one recipient country budget element:{total_count}")

## Total expenditure
response = requests.get('https://datasette.tables.iatistandard.org/iati.json?sql=SELECT+COUNT%28DISTINCT+prefix%29+from+organisation_totalexpenditure')
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Number of organisations with at least one recipient country budget element:{total_count}")

### Download organisation data from the IATI Registry
This will download a list of organisations who have registered, regardless of whether they have published any files.

In [7]:
## Set up a connection to the Registry API and extract the publisher_id from the organization list
x = requests.get('https://iatiregistry.org/api/action/organization_list?all_fields=true')
response_as_json = json.loads(x.text)

publishers = response_as_json["result"]
print (publishers[0])
publisher_id = ['https://iatiregistry.org/api/action/organization_show?id='+x["name"] for x in publishers]

In [13]:
## Run through all of the publishers and accesses their iati_id, the first date they published (if available), the number of packages they publish and their organisation type.
## If an exception is thrown when trying to access publisher_first_publish_date then firstdate is set to "BLANK" to distinguish from empty values that don't throw an exception

## blanks=0
first_pub_date = list()
for x in publisher_id:
    org = requests.get(x)
    response_as_json = json.loads(org.text)
    org = response_as_json["result"]
    try:
        firstdate=org["publisher_first_publish_date"]
    except:
        firstdate = "BLANK"
 ##       blanks+=1
    first_pub_date.append([org["name"], org['publisher_iati_id'], firstdate, org["package_count"], org["publisher_organization_type"]])
    

This creates a dataframe, df_registry_organisations, with the contents of the previous list and the API call that includes their id on the Registry

In [22]:
df_cols =  ['prefix', 'reportingorg_ref', 'registry_first_pub_date', 'package_count', 'organisation_type']
df_registry_organisations = pd.DataFrame(first_pub_date, columns=df_cols)

In [31]:
df_registry_organisations.info()

In [34]:
## Cast columns to correct data types 
df_registry_organisations['prefix'] = df_registry_organisations['prefix'].astype("string")
df_registry_organisations['reportingorg_ref'] = df_registry_organisations['reportingorg_ref'].astype("string")
df_registry_organisations['registry_first_pub_date'] = df_registry_organisations['registry_first_pub_date'].astype("string")
df_registry_organisations['organisation_type'] = df_activity['organisation_type'].astype('category')

In [ ]:
## Save DataFrame to disk as a Parquet file
df_registry_organisations.to_parquet("registry_organisation.parquet", index=False)

In [ ]:
## Delete DataFrame
!rm -rf df_registry_organisations

### Download selected columns from the activity table in IATI Tables: Datasette instance

In [40]:
datasette_url = 'https://datasette.tables.iatistandard.org'
count_url = f"{datasette_url}/iati.json?sql=SELECT+Count(*)+AS+TOTAL+FROM+activity"
response = requests.get(count_url)
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Total rows to fetch:{total_count}")

def fetch_chunk(start_offset, size):
    query_url = f"{datasette_url}/iati.json?sql=select+rowid%2C+_link%2C+_link_activity%2C+prefix%2C+iatiidentifier%2C+plannedstart%2C+actualstart%2C+plannedend%2C+actualend%2C+reportingorg_ref%2C+reportingorg_type%2C+reportingorg_secondaryreporter%2C+reportingorg_narrative%2C+activitystatus_code%2C+activitystatus_codename%2C+activitydate%2C+hierarchy%2C+humanitarian%2C+lang%2C+lastupdateddatetime+from+activity+LIMIT+{size}+OFFSET+{start_offset}"
    response = requests.get(query_url, timeout=120)
    if response.status_code == 200:
        return response.json()['rows']
    return []

# Set up chunks
chunk_size = 20000
offsets = list(range(0, total_count, chunk_size))

# Process in parallel
all_data = []
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    future_to_offset = {executor.submit(fetch_chunk, offset, chunk_size): offset for offset in offsets}
    for future in tqdm(concurrent.futures.as_completed(future_to_offset), total=len(offsets)):
        offset = future_to_offset[future]
        try:
            data = future.result()
            all_data.extend(data)
        except Exception as e:
            print(f"Error with offset {offset}: {e}")

In [43]:
if all_data:
    column_url = f"{datasette_url}/iati.json?sql=select+rowid%2C+_link%2C+_link_activity%2C+prefix%2C+iatiidentifier%2C+plannedstart%2C+actualstart%2C+plannedend%2C+actualend%2C+reportingorg_ref%2C+reportingorg_type%2C+reportingorg_secondaryreporter%2C+reportingorg_narrative%2C+activitystatus_code%2C+activitystatus_codename%2C+activitydate%2C+hierarchy%2C+humanitarian%2C+lang%2C+lastupdateddatetime+from+activity+LIMIT+1"
    column_response = requests.get(column_url)
    columns = column_response.json()['columns']

In [46]:
df_activity = pd.DataFrame(all_data, columns=columns)
df_activity.info()

In [52]:
# Replace empty strings with -1 in the hierarchy column
df_activity['hierarchy'] = df_activity['hierarchy'].replace('', -1)
# Convert hierarchy to numeric before making it categorical
df_activity['hierarchy'] = pd.to_numeric(df_activity['hierarchy'], errors='coerce')
df_activity['hierarchy'] = df_activity['hierarchy'].astype('category')

## convert 'prefix', 'reportingorg_ref','reportingorg_type', 'activitystatus_code','activitystatus_codename','hierarchy' and 'lang' to Categorical datatypes
## to check: whether prefix and reportingorg_ref would be better as Strings
df_activity['prefix'] = df_activity['prefix'].astype('category')
df_activity['reportingorg_ref'] = df_activity['reportingorg_ref'].astype('category')
df_activity['reportingorg_type'] = df_activity['reportingorg_type'].astype('category')
df_activity['activitystatus_code'] = df_activity['activitystatus_code'].astype('category')
df_activity['activitystatus_codename'] = df_activity['activitystatus_codename'].astype('category')
df_activity['hierarchy'] = df_activity['hierarchy'].astype('category')
df_activity['lang'] = df_activity['lang'].astype('category')
## Convert 'rowid', '_link', '_link_activity', 'iatiidentifier', 'reportingorg_narrative' to string
df_activity['rowid'] = df_activity['rowid'].astype('str')
df_activity['_link'] = df_activity['_link'].astype('str')
df_activity['_link_activity'] = df_activity['_link_activity'].astype('str')
df_activity['iatiidentifier'] = df_activity['iatiidentifier'].astype('str')
df_activity['reportingorg_narrative'] = df_activity['reportingorg_narrative'].astype('str')
## convert 'plannedstart', 'actualstart','plannedend','actualend','lastupdateddatetime' to datetime
## Using mixed format to handle various datetime formats and coerce errors to NaT
df_activity['plannedstart'] = pd.to_datetime(df_activity['plannedstart'], format='mixed', utc=True, errors='coerce')
df_activity['actualstart'] = pd.to_datetime(df_activity['actualstart'], format='mixed', utc=True, errors='coerce')
df_activity['plannedend'] = pd.to_datetime(df_activity['plannedend'], format='mixed', utc=True, errors='coerce')
df_activity['actualend'] = pd.to_datetime(df_activity['actualend'], format='mixed', utc=True, errors='coerce')
df_activity['lastupdateddatetime'] = pd.to_datetime(df_activity['lastupdateddatetime'], format='mixed', utc=True, errors='coerce')
## convert 'reportingorg_secondaryreporter','humanitarian' to bool
df_activity['reportingorg_secondaryreporter'] = df_activity['reportingorg_secondaryreporter'].astype('bool')
df_activity['humanitarian'] = df_activity['humanitarian'].astype('bool')

In [55]:
# Save to parquet
df_activity.to_parquet("datasette_activities.parquet", index=False)

In [ ]:
## Delete the DataFrame
!rm -rf df_activity

### Download selected columns from the transaction table in IATI Tables: Datasette instance

This section connects to the Datasette instance that hosts the IATI transaction table and determines the overall rowid range that needs to be downloaded. It runs a simple SQL query to retrieve the minimum and maximum rowid values, which gives an approximate count of all transaction records. Knowing this range is essential for splitting the download into manageable chunks later in the workflow.

In [127]:
# Base URL for the IATI Datasette instance
# All API requests for the transaction table will use this base path.
BASE = "https://datasette.tables.iatistandard.org/iati"

# Use a requests Session for connection reuse and improved performance,
# especially helpful when making many sequential download requests.
SESSION = requests.Session()

# SQL query to retrieve the minimum and maximum rowid values in the 'trans' table.
# This gives us the full range of available rows.
sql = "select min(rowid) as min_id, max(rowid) as max_id from trans"

# Construct the URL for the Datasette JSON endpoint and URL-encode the SQL query.
url = f"{BASE}.json?sql={quote(sql)}"

# Send the HTTP request to the server with a 120-second timeout.
r = SESSION.get(url, timeout=120)

# Raise an exception if the request returned an HTTP error (4xx/5xx).
r.raise_for_status()

# Extract min_id and max_id from the returned JSON response.
min_id, max_id = r.json()["rows"][0]

# Print the discovered rowid range and the approximate number of rows.
print("rowid range:", min_id, max_id, "→ approx rows:", max_id - min_id + 1)

This section figures out a safe chunk size for downloading the data by using a small sample of rows. It first defines the list of columns we actually want to keep (to avoid downloading unnecessary data) and builds a SQL query that selects only those columns for a small range of rowid values. It then downloads this sample as a streamed CSV, measures how many bytes it uses, and calculates an approximate bytes per row. Using that estimate and a target file size (around 90 MB), it computes a suggested ROWS_PER_CHUNK value

In [130]:
# Estimate a safe ROWS_PER_CHUNK using a small sample
# Define a list of columns to keep, reducing file size and speeding up downloads.
# Set KEEP_COLS = None to download all columns instead.
KEEP_COLS = ["_link_activity","prefix","iatiidentifier","reportingorg_ref","humanitarian",
             "transactiontype_code","transactiondate","transactiondate_isodate","value",
             "value_currency","value_valuedate","providerorg_ref","providerorg_provideractivityid",
             "providerorg_type","providerorg_typename","receiverorg_ref",
             "receiverorg_receiveractivityid","receiverorg_type","receiverorg_typename",
             "value_usd","sector_code"]

# Function to build the SQL SELECT clause based on which columns we want to keep.
def select_clause():
    return "*" if KEEP_COLS is None else ", ".join([f'"{c}"' for c in KEEP_COLS])

print(select_clause)

# Number of rows to download as a sample; small enough to be fast but large enough for estimation.
SAMPLE_ROWS = 25_000  # small, fast sample

# Build the SQL query to fetch the sample rows from the 'trans' table.
sample_sql = f"""
select rowid as _rowid_, {select_clause()}
from trans
where rowid between {min_id} and {min_id + SAMPLE_ROWS - 1}
order by rowid
""".strip()

print(sample_sql)

# Construct the Datasette CSV streaming URL for the sample query.
sample_url = f"{BASE}.csv?sql={quote(sample_sql)}&_stream=on"

# Download the sample as a streamed CSV and count the total number of bytes received.
sample_bytes = 0
with SESSION.get(sample_url, stream=True, timeout=600) as resp:
    resp.raise_for_status()
    # Iterate over the streamed chunks (1 MB each).
    for chunk in resp.iter_content(chunk_size=1024 * 1024):
        sample_bytes += len(chunk)

# Compute bytes per row based on the sample size.
bytes_per_row = max(1, sample_bytes // max(1, SAMPLE_ROWS))

# Set a target file size (MB) to stay below the server's limit (~100 MB).
target_mb = 90

# Calculate a safe number of rows per chunk based on estimated bytes per row.
ROWS_PER_CHUNK = max(10_000, (target_mb * 1024 * 1024) // bytes_per_row)

# Print diagnostic information about the sample and calculated chunk size.
print(f"Sample bytes: {sample_bytes:,}")
print(f"Estimated bytes/row: {bytes_per_row:,}")
print(f"Suggested ROWS_PER_CHUNK ≈ {ROWS_PER_CHUNK:,}")

This section performs the actual downloading of the transaction data in CSV chunks, using the previously calculated ROWS_PER_CHUNK size. It creates a directory to store the downloaded files and loops through the full rowid range, generating SQL queries that fetch only the required rows and selected columns for each chunk. Each request streams the CSV data directly to disk to avoid memory issues.

In [133]:
# Download CSV chunks (plain CSV, streamed)

# Directory where all CSV chunk files will be stored
CHUNK_DIR = Path("trans_chunks")
CHUNK_DIR.mkdir(exist_ok=True)

# Use the previously estimated rows-per-chunk value
ROWS_PER_CHUNK = int(263_608)  # from our estimate; can be tweaked smaller if any files are truncated
print(ROWS_PER_CHUNK)

# List to store the file paths of all downloaded CSV chunks
paths = []

# Start from the minimum rowid discovered earlier
start = min_id

# Loop until we reach the maximum rowid
while start <= max_id:
    # Determine the end rowid for this chunk, not exceeding max_id
    end = min(start + ROWS_PER_CHUNK - 1, max_id)

    # SQL query to select a slice of rows (by rowid range) and specific columns
    sql = f"select rowid as _rowid_, _link_activity, prefix, iatiidentifier, reportingorg_ref, humanitarian, transactiontype_code, transactiondate, transactiondate_isodate, value, value_currency, value_valuedate, value_usd from trans where rowid between {start} and {end} order by rowid"

    # Build the CSV streaming URL for this chunk
    url = f"{BASE}.csv?sql={quote(sql)}&_stream=on"

    # Output file path for this chunk, named with its start and end rowid
    out = CHUNK_DIR / f"trans_{start}_{end}.csv"

    # Optional logic (currently commented out) to skip downloading if the file already exists
    #    if out.exists():
    #        print(f"exists, skipping {out.name}")
    #        paths.append(str(out))
    #    else:
    #        # Force identity encoding so we save plain CSV (not HTTP-gzipped bytes)

    # Download the chunk as a streamed CSV and write it directly to disk
    with SESSION.get(url, headers={"Accept-Encoding": "identity"}, stream=True, timeout=1200) as resp:
        resp.raise_for_status()
        with open(out, "wb") as f:
            shutil.copyfileobj(resp.raw, f)

    # Print confirmation with the saved file name and its size in MB
    print(f"saved {out.name}  ({out.stat().st_size/1_048_576:.1f} MB)")

    # Record the path of the saved file
    paths.append(str(out))

    # Move the start pointer to the next chunk
    start = end + 1

# Quick check: how many chunk files were downloaded
len(paths)

This section checks that every downloaded CSV file is complete and not truncated. It rebuilds or reuses the list of chunk file paths, then for each file it reads the _rowid_ column to find the last rowid actually present in the file. It compares that value with the expected end rowid encoded in the filename (e.g. trans_1_263608.csv should end at 263608). If a file doesn’t reach its expected final rowid, it is flagged as potentially truncated.

In [136]:
# Integrity check: confirm each CSV reaches its expected end rowid

# If we restarted the kernel or lost the 'paths' variable,
# rebuild the list of CSV chunk file paths from the directory.
try:
    paths
except NameError:
    CHUNK_DIR = Path("trans_chunks")
    paths = [str(p) for p in sorted(CHUNK_DIR.glob("trans_*.csv"))]

# Extract the expected rowid range from the filename
# Example filename: trans_1_263608.csv → start=1, end=263608
def range_from_filename(p):
    name = Path(p).name
    start_i = int(name.split("_")[1])
    end_i = int(name.split("_")[2].split(".")[0])
    return start_i, end_i

# Read a CSV file and return the last _rowid_ value found in it
def last_rowid_in_csv(path):
    last = None
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        try:
            idx = header.index("_rowid_")  # find the index of the _rowid_ column
        except ValueError:
            raise RuntimeError(f"_rowid_ column not found in {path}")

        # Iterate through rows to capture the last valid rowid
        for row in reader:
            if len(row) > idx and row[idx] != "":
                try:
                    last = int(row[idx])
                except ValueError:
                    pass  # skip rows with non-integer rowid values

    return last

# Check each file and collect those that appear truncated
truncated = []
for p in paths:
    s, e = range_from_filename(p)   # expected start and end rowid
    last = last_rowid_in_csv(p)     # actual last rowid in CSV

    # If no rowid found, or last rowid is smaller than expected, flag it
    if last is None or last < e:
        print(f"TRUNCATED? {Path(p).name}  last_rowid={last}  expected_end={e}")
        truncated.append((s, e))

# Summary of integrity check results
print("Files flagged as truncated:", len(truncated))

Combine all the trans CSV files into a dataframe

This section combines all the previously downloaded transaction CSV chunks into a single pandas dataframe. It first collects the list of trans_*.csv files from the trans_chunks directory, then reads each CSV into pandas and concatenates them into one large dataframe called df_trans. This gives you a unified in-memory view of the entire transaction table, instead of many separate files on disk. The printed shape and row count act as a quick sanity check that all expected data has been loaded and combined correctly

In [139]:
# Get all the CSV files from the trans_chunks directory
# These are the chunk files downloaded in the previous step.
CHUNK_DIR = Path("trans_chunks")
paths = [str(p) for p in sorted(CHUNK_DIR.glob("trans_*.csv"))]

# Display how many chunk files were found
print(f"Found {len(paths)} CSV files to combine")

# Read all CSV chunks into pandas dataframes and concatenate them
# ignore_index=True creates a continuous index across all rows
df_trans = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

# Print the resulting dataframe dimensions for validation
print(f"\nCombined dataframe shape: {df_trans.shape}")

# Print the total number of rows as another quick sanity check
print(f"Total rows: {len(df_trans):,}")

# Request display of the first few rows (actual display happens outside print)
print(f"\nFirst few rows:")

Convert datatypes

This section prepares the combined dataframe for efficient storage and analysis by converting each column to the correct data type

In [142]:
# Convert datatypes

# Cast row identifiers and link fields to string type
df_trans['_rowid_'] = df_trans['_rowid_'].astype('str')
df_trans['_link_activity'] = df_trans['_link_activity'].astype('str')
df_trans['iatiidentifier'] = df_trans['iatiidentifier'].astype('str')

# Convert several columns to category dtype for memory efficiency
df_trans['prefix'] = df_trans['prefix'].astype('category')
df_trans['reportingorg_ref'] = df_trans['reportingorg_ref'].astype('category')
df_trans['transactiontype_code'] = df_trans['transactiontype_code'].astype('str').astype('category')
df_trans['value_currency'] = df_trans['value_currency'].astype('category')

# Convert date-like strings to pandas datetime objects
# - format='mixed' lets pandas detect various date formats
# - utc=True standardizes to UTC timezone
# - errors='coerce' converts invalid dates to NaT rather than raising errors
df_trans['transactiondate'] = pd.to_datetime(df_trans['transactiondate'], format='mixed', utc=True, errors='coerce')
df_trans['transactiondate_isodate'] = pd.to_datetime(df_trans['transactiondate_isodate'], format='mixed', utc=True, errors='coerce')
df_trans['value_valuedate'] = pd.to_datetime(df_trans['value_valuedate'], format='mixed', utc=True, errors='coerce')

In [145]:
df_trans.info()

Save as Parquet

This section writes the fully cleaned and typed transaction dataframe to a Parquet file.

In [151]:
# Save to parquet

# Write the cleaned and typed dataframe to a Parquet file.
# index=False prevents pandas from writing the dataframe index as a separate column.
df_trans.to_parquet("datasette_transaction.parquet", index=False)

Delete variables and cleanup

This section removes temporary data and frees up storage space after the processing workflow is complete. It deletes the trans_chunks directory, which contains all intermediate CSV files, since these are no longer needed once the combined Parquet file has been created.

In [154]:
## Delete variables and DataFrame

# Remove the directory containing all downloaded CSV chunks.
# These intermediate files are no longer needed after saving the Parquet file.
!rm -rf trans_chunks

# Remove the in-memory dataframe to free up system memory.
!rm -rf df_trans

### Download selected columns from the budget table in IATI Tables: Datasette instance

In [100]:
datasette_url = 'https://datasette.tables.iatistandard.org'
count_url = f"{datasette_url}/iati.json?sql=SELECT+Count(*)+AS+TOTAL+FROM+budget"
response = requests.get(count_url)
total_count = response.json()['rows'][0][0] if response.status_code == 200 else None
print(f"Total rows to fetch:{total_count}")

def fetch_chunk(start_offset, size):
    query_url = f"{datasette_url}/iati.json?sql=select+rowid%2C+dataset%2C+_link%2C+_link_activity%2C+prefix%2C+iatiidentifier%2C+reportingorg_ref%2C+type%2C+typename%2C+status%2C+statusname%2C+periodstart%2C+periodstart_isodate%2C+periodend%2C+periodend_isodate+from+budget+LIMIT+{size}+OFFSET+{start_offset}"
    response = requests.get(query_url, timeout=120)
    if response.status_code == 200:
        return response.json()['rows']
    return []

# Set up chunks
chunk_size = 20000
offsets = list(range(0, total_count, chunk_size))

# Process in parallel
all_data = []
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    future_to_offset = {executor.submit(fetch_chunk, offset, chunk_size): offset for offset in offsets}
    for future in tqdm(concurrent.futures.as_completed(future_to_offset), total=len(offsets)):
        offset = future_to_offset[future]
        try:
            data = future.result()
##            print(data)
            all_data.extend(data)
        except Exception as e:
            print(f"Error with offset {offset}: {e}")

In [103]:
if all_data:
     column_url = f"{datasette_url}/iati.json?sql=select+rowid%2C+dataset%2C+_link%2C+_link_activity%2C+prefix%2C+iatiidentifier%2C+reportingorg_ref%2C+type%2C+typename%2C+status%2C+statusname%2C+periodstart%2C+periodstart_isodate%2C+periodend%2C+periodend_isodate+from+budget+LIMIT+1"
     column_response = requests.get(column_url)
     columns = column_response.json()['columns']
     print(columns)
else:
    print("no data")

In [106]:
df_budget = pd.DataFrame(all_data, columns=columns)

In [112]:
## Convert datatypes
## Convert string columns
df_budget['rowid'] = df_budget['rowid'].astype('str')
df_budget['dataset'] = df_budget['dataset'].astype('str')
df_budget['_link'] = df_budget['_link'].astype('str')
df_budget['_link_activity'] = df_budget['_link_activity'].astype('str')
df_budget['iatiidentifier'] = df_budget['iatiidentifier'].astype('str')

## Convert categorical columns (sanity check prefix and reportingorg_ref)
df_budget['prefix'] = df_budget['prefix'].astype('category')
df_budget['reportingorg_ref'] = df_budget['reportingorg_ref'].astype('category')
df_budget['type'] = df_budget['type'].astype('category')
df_budget['typename'] = df_budget['typename'].astype('category')
df_budget['status'] = df_budget['status'].astype('category')
df_budget['statusname'] = df_budget['statusname'].astype('category')

# Convert date columns
df_budget['periodstart'] = pd.to_datetime(df_budget['periodstart'], format='mixed', utc=True, errors='coerce')
df_budget['periodstart_isodate'] = pd.to_datetime(df_budget['periodstart_isodate'], format='mixed', utc=True, errors='coerce')
df_budget['periodend'] = pd.to_datetime(df_budget['periodend'], format='mixed', utc=True, errors='coerce')
df_budget['periodend_isodate'] = pd.to_datetime(df_budget['periodend_isodate'], format='mixed', utc=True, errors='coerce')

In [115]:
df_budget.info()

In [118]:
## Save to parquet
df_budget.to_parquet("datasette_budget.parquet", index=False)

In [121]:
## clean up
del df_budget

### Download data from the timeliness table on the IATI Dashboard, append the warning flags 

#### Alternative method that scrapes the Timeliness page for the prefix, reporting-org name, frequency and flags

In [4]:
## Scrape the timeliness frequency table at https://dashboard.iatistandard.org/publishing-statistics/timeliness-frequency/
## Capture the Reporting Org's name, prefix (from the Registry) and data-severity value
## This can be updated/simplified once the dashboard's CSV download includes the data-severity values

def scrape_iati_timeliness_table(url):

    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"Error fetching the URL: {e}"

    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the frequency table by its class
    table = soup.find('table', {'class': 'table iati-table__table'})
    # Return an error message if the table is not found
    if not table:
        return "Table 'table iati-table__table' not found on the page."
    
    data = []
    # Loop through the table and extract the reporting org's name, first published, prefix, frequency and data-severity value
    for row in table.find('tbody').find_all('tr'):
        row_data = {}
        cells = row.find_all('td')
        links = row.find_all('a', href=True)
        flags = row.find_all('td', {'data-severity': True})
       
        ## 'Reporting Org Name' is the first cell
        row_data['Reporting Org Name'] = cells[0].text.strip()

        ## First Published is in thge second cell
        row_data['First published'] = cells[1].text.strip()
       
        ## We can construct the prefix from the link in the first cell
        prefix = (links[0]['href'])[12:]
        prefix = prefix[:-1]
        row_data['prefix'] = prefix

         ## 'Frequency' is in the seventeenth cell
        row_data['Frequency']= cells[16].text.strip()

        data_severity = flags[0]
        row_data['data-severity'] = data_severity.get('data-severity')

        data.append(row_data)

    df = pd.DataFrame(data)
    return df

# URL of the page to scrape
url_to_scrape = "https://dashboard.iatistandard.org/publishing-statistics/timeliness-frequency/"

# Scrape the data and get the DataFrame
df_scraped_timeliness = scrape_iati_timeliness_table(url_to_scrape)

df_scraped_timeliness.info()

In [25]:
df_scraped_timeliness

In [28]:
## Save to parquet
df_scraped_timeliness.to_parquet("scraped_timeliness.parquet", index=False)

In [ ]:
## Delete DataFrame
!rm -rf df_scraped_timeliness

### Download data from the timelag page on the dashboard

In [136]:
## Quick method that doesn't capture flags as we will be merging timelag into the summary dataframe in Analysis where it is already captured
DASH_URL = "https://dashboard.iatistandard.org/generated/data/csv/timeliness_timelag.csv"
df_timelag  = pd.read_csv(DASH_URL)
df_timelag = df_timelag[["Publisher Registry Id", "Time lag"]]
df_timelag = df_timelag.rename(columns={'Publisher Registry Id': 'prefix'})
df_timelag

In [ ]:
# Cast 'Time lag' column to category
df_timelag['Time lag'] = df_timelag['Time lag'].astype('category')

In [139]:
# Save to disk as CSV file
df_timelag.to_csv("timeliness_timelag_without_flags.csv", index=False)

In [ ]:
# Delete DataFrame
!rm -rf df_timelag

### Download comprehensiveness_current_activities from the Dashboard

In [10]:
url = "https://dev2.dashboard.iatistandard.org/stats/current/aggregated/comprehensiveness_current_activities.json"
with urllib.request.urlopen(url) as response:
    data = json.load(response)
df_current = pd.DataFrame(data.items(), columns=["iatiidentifier", "flag"])

In [13]:
df_current

In [19]:
df_current['Active'] = [0 if x == 0 else 1 for x in df_current['flag']]

In [22]:
df_current

In [28]:
df_current = df_current.drop('flag', axis=1)
df_current

In [34]:
df_current.to_csv('df_current.csv')

In [ ]:
!rm -rf df_current

### Download activity date data from IATI Tables

Downloading actual and planned, start and end, in own table - to keep data manageable.

In [37]:
%%nql act_starts = DF

SELECT DISTINCT 
    reportingorg_ref, 
    iatiidentifier, 
    typename, 
    isodate 
FROM activitydate 
WHERE typename = 'Actual start'

In [40]:
%%nql plan_starts = DF

SELECT DISTINCT 
    reportingorg_ref, 
    iatiidentifier, 
    typename, 
    isodate 
FROM activitydate 
WHERE typename = 'Planned start'

In [43]:
%%nql act_ends = DF

SELECT DISTINCT 
    reportingorg_ref, 
    iatiidentifier, 
    typename, 
    isodate 
FROM activitydate 
WHERE typename = 'Actual end'

In [46]:
%%nql plan_ends = DF

SELECT DISTINCT 
    reportingorg_ref, 
    iatiidentifier, 
    typename, 
    isodate 
FROM activitydate 
WHERE typename = 'Planned End'

In [49]:
#concat to one dataframe of dates
act_dates = pd.concat([plan_starts,act_starts,plan_ends,act_ends], ignore_index=True)
act_dates

In [52]:
!rm -rf "downloads/act_dates.csv"
 
#save data
act_dates.to_csv("downloads/act_dates.csv",index=False)

### Download weekly update counts - dashboard backend

Get list of orgs to construct urls.

In [58]:
#get list of orgs from the dashboard
!wget https://dashboard.iatistandard.org/generated/data/csv/publishers.csv

orgs = pd.read_csv("publishers.csv")

!rm -rf publishers.csv 

Download most recent transaction date data - will take a while to run.

In [61]:
most_recent_trans = {}

for id in orgs['Publisher Registry Id']:
    org_url = "https://dashboard.iatistandard.org/stats/gitaggregate-publisher-dated/" + id + "/most_recent_transaction_date.json"
    with urllib.request.urlopen(org_url) as url:
        most_recent_trans[id] = json.load(url)

Save dictionary

In [64]:
!rm -rf "downloads/most_recent_trans.pkl"

with open("downloads/most_recent_trans.pkl", 'wb') as f:
    pickle.dump(most_recent_trans, f)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=f9058ed8-51a0-4f04-bd96-e60901d82949' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>